# Antarctic MHW drivers — Richaud method

A deliberate parallel to `04_polar_mhw_arctic.ipynb`, so the two hemispheres can be compared
figure for figure. Same tiling, same event bookkeeping, same driver ranking, same two headline
figures — the Antarctic instead of the Arctic.

**This is not a replacement for NB03.** NB03 is our approach: 2.5°×2.5° geographic boxes,
composites of the mean anomaly per term. This notebook is Benjamin's: 20×20 grid-cell tiles,
and *counts of events* by which term ranks first or second. Keeping them separate means a
disagreement between them is informative rather than confusing.

## What is the same as NB04

- **20×20 grid-cell tiles**, kept when ≥75% of their cells are ocean (≥300 of 400) — the
  criterion from Richaud et al.'s tiling, chosen so adjacent tiles are close to independent.
- **MHW detection**: Hobday et al. (2016), `MIN_DUR = 5`, `MAX_GAP = 2`, against
  `om2_025_MLT_clim.nc` / `om2_025_MLT_thresh.nc`.
- **Budget climatology** from `mhw3d.best_practice.compute_climatology` with default
  parameters — NB04 calls it with no arguments, and `00_budget_clim_daily.ipynb` uses the same
  settings, so the two hemispheres rest on the same construction.
- **`ds_events`**: one dataset of (events × tiles × phase), built by `box_df2xr`.
- **Driver ranking** and the two figures ported from NB04 cells 39–44.

## What the Antarctic lets us drop

NB04 carries a lot of machinery for the tripolar grid north of ~65°N. South of 60°S the
ACCESS-OM2 grid is regular, so:

- **No meshgrid file.** `yt_ocean` and `xt_ocean` *are* latitude and longitude, so
  `geolat_t`/`geolon_t` are unnecessary.
- **No `areacello` file.** On a regular grid cell area ∝ cos(latitude), and constant factors
  cancel in a weighted mean, so `cos(yt_ocean)` gives numerically the same area-weighted mean
  that NB04 gets from `areacello`.
- **No `preprocess_60N`**, and no north-pole special-casing of tile geometry.

Tiles are still built in index space rather than degrees, exactly as NB04 does — that is what
keeps the tile statistics comparable between hemispheres.

## One difference worth arguing about

NB04 ranks drivers with `.rank(axis=1, ascending=False)` on the **signed** anomaly, so rank 1
is the most positive term. During onset that is the strongest warming contributor, which is
what you want. During *decline* it is the term most strongly opposing the decline — the
strongest cooling term gets the *last* rank, not the first.

NB03 instead takes the largest |anomaly| regardless of sign.

`RANK_BY` below selects between them. It defaults to `'signed'`, matching NB04, because the
point of this notebook is comparability — but the decline ranking is worth raising with
Benjamin before either set of numbers goes in the chapter.

In [ ]:
import os
import pickle

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.path as mpath
from matplotlib.ticker import PercentFormatter
import cartopy.crs as ccrs
import cartopy.feature as cfeature

from dask.distributed import Client

%matplotlib inline


In [ ]:
# processes=True + threads_per_worker=1 avoids netCDF4/HDF5 thread-safety races
# when many workers read chunks from the same open_mfdataset files concurrently.
client = Client(processes=True, threads_per_worker=1, n_workers=6)
client

## 1. Configuration

In [ ]:
# ── Domain ────────────────────────────────────────────────────────────────────
# South of 60°S, mirroring NB04's north of 60°N.
LAT_MIN      = -80.0
LAT_MAX      = -60.0
BOX_SIZE     = 20      # tile edge in GRID CELLS (not degrees), as in NB04
MIN_OCN_FRAC = 0.75    # keep tiles with >= 0.75 * 400 = 300 ocean cells

# ── Data ──────────────────────────────────────────────────────────────────────
base         = '/g/data/av17/access-nri/OM2/025deg_jra55_iaf_cycle6_online_mlt/'
BUDGET_SUBDIR = 'post_processed_diags/mlt_budget_online_stavg/'
START_OUTPUT = 357     # 2010
END_OUTPUT   = 366     # 2019
outputs      = list(range(START_OUTPUT, END_OUTPUT + 1))

OUTPUT_DIR   = '/scratch/m35/nm5072/Polar_MHWs/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Daily budget climatology from 00_budget_clim_daily.ipynb. NB04 computes its own
# with the same mhw3d call and defaults; we read ours so both hemispheres use one
# construction. Check the baseline_years attribute matches what NB04 used.
BUDGET_CLIM_FILE = OUTPUT_DIR + 'mlt_budget_clim_daily_336-365_global.nc'

p2figs = OUTPUT_DIR + 'figures/'
os.makedirs(p2figs, exist_ok=True)

# ── MHW detection ─────────────────────────────────────────────────────────────
MIN_DUR = 5
MAX_GAP = 2

# ── Driver ranking ────────────────────────────────────────────────────────────
# 'signed' : rank 1 = most positive anomaly            (NB04's choice)
# 'abs'    : rank 1 = largest |anomaly|                (NB03's choice)
# See the note in the header — these disagree during decline.
RANK_BY = 'signed'

# ── Dask / I/O ────────────────────────────────────────────────────────────────
chunks2D    = {'time': 365, 'yt_ocean': 216, 'xt_ocean': 240}
SEC_PER_DAY = 86400.0

# ── Budget terms ──────────────────────────────────────────────────────────────
TERM_MAP = {
    'MLT tendency'   : 'mlt_tendency',
    'Surface Flux'   : 'surf_to_ML',
    'Advection'      : 'advection',
    'Vertical mixing': 'vert_mixing',
    'Entrainment'    : 'entrainment',
}
BUDGET_TERMS = list(TERM_MAP.keys())

# ── Plotting ──────────────────────────────────────────────────────────────────
# NB04 plots with ultraplot; these are the same four colours by their XKCD names,
# which matplotlib understands directly, so the two hemispheres' figures match
# without the extra dependency.
DRIVER_COLORS = ['xkcd:ocher', 'xkcd:cobalt', 'xkcd:wine red', 'xkcd:grass']
DRIVER_TITLES = ['Surface\nHeat Flux', 'Lateral\nHeat Flux',
                 'Vertical\nMixing', 'Entrainment']
DRIVER_LEGEND = ['Surface heat flux', 'Lateral heat flux',
                 'Vertical mixing', 'Entrainment']
DRIVER_TICKS  = ['Surface', 'Lateral', 'Vertical mixing', 'Entrainment']
SMAX          = 320.0     # scatter area for a fraction of 1.0, shared by all panels

print('Domain : {0:.0f}S to {1:.0f}S, {2:d}x{2:d} grid-cell tiles'.format(
    abs(LAT_MIN), abs(LAT_MAX), BOX_SIZE))
print('Years  : outputs {0}-{1} (2010-2019)'.format(START_OUTPUT, END_OUTPUT))
print('Ranking: {0}'.format(RANK_BY))

## 2. Load data

Everything is lazy until section 4. `xt_ocean` is left unchunked so no chunk boundary lands on
the ACCESS-OM2 grid seam at ~80°E — tiles are contiguous index blocks and a tile spanning the
seam would otherwise straddle chunks.

In [ ]:
mlt_files = [base + 'output{0:03d}/ocean/ocean_daily.nc'.format(o) for o in outputs]
budget_files = [base + BUDGET_SUBDIR +
                'mlt_budget_stavg_daily_online_output{0:03d}.nc'.format(o) for o in outputs]

vars_raw = ['mlt_tendency', 'advection', 'vert_mixing',
            'entrainment', 'surface_flux', 'sw_pen', 'residual']

ds_wide = xr.open_mfdataset(
    mlt_files, decode_times=True, chunks=chunks2D,
    combine='nested', concat_dim='time',
    data_vars=['temp_in_mld', 'mld'], parallel=True, decode_timedelta=False,
).sel(yt_ocean=slice(LAT_MIN, LAT_MAX))
temp_wide = ds_wide['temp_in_mld'] / 1035

budget_wide = xr.open_mfdataset(
    budget_files, decode_times=True, chunks=chunks2D,
    combine='nested', concat_dim='time',
    parallel=True, decode_timedelta=False,
).sel(yt_ocean=slice(LAT_MIN, LAT_MAX))[vars_raw]
budget_wide['surf_to_ML'] = budget_wide['surface_flux'] + budget_wide['sw_pen']

clim_wide = xr.open_dataset(
    base + 'post_processed_diags/om2_025_MLT_clim.nc'
).sel(yt_ocean=slice(LAT_MIN, LAT_MAX))
thresh_wide = xr.open_dataset(
    base + 'post_processed_diags/om2_025_MLT_thresh.nc'
).sel(yt_ocean=slice(LAT_MIN, LAT_MAX))

budget_clim_wide = xr.open_dataset(BUDGET_CLIM_FILE).sel(
    yt_ocean=slice(LAT_MIN, LAT_MAX))

print('MLT / budget loaded (lazy): {0}'.format(dict(budget_wide.sizes)))
print('Budget climatology : {0}'.format(BUDGET_CLIM_FILE))
print('  baseline : {0} (outputs {1})'.format(
    budget_clim_wide.attrs.get('baseline_years', 'UNKNOWN'),
    budget_clim_wide.attrs.get('baseline_outputs', '?')))
print('  NB04 used 2010-2019 — if these differ, the hemispheres are not directly comparable.')

## 3. Tiles

NB04's `generate_tiles_boxes`, simplified for a regular grid. Tiles are contiguous 20×20 blocks
of grid cells, numbered in index space; a tile is kept when at least `MIN_OCN_FRAC` of its 400
cells are ocean. Centres come straight from `yt_ocean`/`xt_ocean` rather than from a meshgrid
file, because south of 60°S those *are* latitude and longitude.

In [ ]:
def polar_ax(ax, boundinglat=LAT_MAX):
    '''South-polar axes with a circular boundary, matching NB04's 'splaea' panels.'''
    ax.set_extent([-180, 180, -90, boundinglat], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.OCEAN, facecolor='0.82', zorder=0)
    ax.add_feature(cfeature.LAND, facecolor='0.6', edgecolor='k',
                   linewidth=0.4, zorder=2)
    ax.coastlines(linewidth=0.5, zorder=3)
    ax.gridlines(linestyle=':', linewidth=0.4, alpha=0.5, zorder=4)
    th = np.linspace(0, 2 * np.pi, 200)
    ax.set_boundary(mpath.Path(np.vstack([np.sin(th), np.cos(th)]).T * 0.5 + [0.5, 0.5]),
                    transform=ax.transAxes)
    return ax


mask_file = base + 'output{0:03d}/ocean/ocean_daily.nc'.format(START_OUTPUT)
mask_month = (
    xr.open_dataset(mask_file, decode_times=False)
    .isel(time=slice(0, 31))
    .sel(yt_ocean=slice(LAT_MIN, LAT_MAX))
)
ocean_full = (mask_month['temp_in_mld'] / 1035).notnull().mean('time').compute()
print('Mask computed from {0}'.format(mask_file))


def generate_tiles(mask, box_size=BOX_SIZE, ratio_gc=MIN_OCN_FRAC):
    """Tile the domain into box_size x box_size blocks of grid cells.

    Returns a list of dicts with the index bounds of each kept tile, its centre in
    index space (x_center / y_center, used for the tile id) and in degrees
    (lon_center / lat_center), and its ocean fraction.
    """
    ny, nx = mask.sizes['yt_ocean'], mask.sizes['xt_ocean']
    ngcmin = box_size * box_size * ratio_gc
    is_ocean = (mask > 0).values

    boxes = []
    for y0 in range(0, ny, box_size):
        y1 = min(y0 + box_size, ny)
        for x0 in range(0, nx, box_size):
            x1 = min(x0 + box_size, nx)
            n_ocean = int(is_ocean[y0:y1, x0:x1].sum())
            if n_ocean < ngcmin:
                continue
            lat = float(mask.yt_ocean[y0:y1].mean())
            lon = float(mask.xt_ocean[x0:x1].mean())
            boxes.append({
                'y0': y0, 'y1': y1, 'x0': x0, 'x1': x1,
                'y_center': (y0 + y1 - 1) / 2.0,
                'x_center': (x0 + x1 - 1) / 2.0,
                'lat_center': lat,
                'lon_center': lon % 360.0,
                'ocean_frac': n_ocean / float((y1 - y0) * (x1 - x0)),
            })
    return boxes


def _box_slice(da, box):
    """Tiles are contiguous index blocks, so this is a plain isel — no seam handling."""
    return da.isel(yt_ocean=slice(box['y0'], box['y1']),
                   xt_ocean=slice(box['x0'], box['x1']))


def _weights(da):
    """Area weights. On this regular grid cell area is proportional to cos(lat), and
    constant factors cancel in a weighted mean, so this equals NB04's areacello
    weighting numerically."""
    return np.cos(np.deg2rad(da['yt_ocean']))


def _box_mean(da, box):
    sub = _box_slice(da, box)
    return sub.weighted(_weights(sub)).mean(('xt_ocean', 'yt_ocean'))


all_boxes = generate_tiles(ocean_full)
box_ids = ['x{0:.0f}_y{1:.0f}'.format(b['x_center'], b['y_center']) for b in all_boxes]

n_y = int(np.ceil(ocean_full.sizes['yt_ocean'] / BOX_SIZE))
n_x = int(np.ceil(ocean_full.sizes['xt_ocean'] / BOX_SIZE))
print('Grid: {0} x {1} cells -> {2} x {3} candidate tiles'.format(
    ocean_full.sizes['yt_ocean'], ocean_full.sizes['xt_ocean'], n_y, n_x))
print('Tiles kept (>= {0:.0f} ocean cells): {1}'.format(
    BOX_SIZE * BOX_SIZE * MIN_OCN_FRAC, len(all_boxes)))
print('Latitude span of tile centres: {0:.1f} to {1:.1f}'.format(
    min(b['lat_center'] for b in all_boxes),
    max(b['lat_center'] for b in all_boxes)))

In [ ]:
# Context map — where the tiles are
fig = plt.figure(figsize=(7, 6.5))
ax = plt.axes(projection=ccrs.SouthPolarStereo())
polar_ax(ax)
sc = ax.scatter([b['lon_center'] for b in all_boxes],
                [b['lat_center'] for b in all_boxes],
                c=[b['ocean_frac'] for b in all_boxes],
                s=45, cmap='Blues', vmin=MIN_OCN_FRAC, vmax=1.0,
                edgecolor='k', linewidth=0.3,
                transform=ccrs.PlateCarree(), zorder=5)
plt.colorbar(sc, ax=ax, label='Ocean fraction', shrink=0.65, pad=0.06)
ax.set_title('{0} tiles of {1}x{1} grid cells'.format(len(all_boxes), BOX_SIZE),
             fontsize=12)
plt.tight_layout()


## 4. Pre-compute tile means

One area-weighted mean per tile for the MLT, its climatology and threshold, and each budget
term. Checkpointed per y-band so a crash costs one band rather than the whole run.

In [ ]:
y_band_centers = sorted(set(b['y_center'] for b in all_boxes))
y_bands = {yc: [(bid, b) for bid, b in zip(box_ids, all_boxes)
                if abs(b['y_center'] - yc) < 0.01]
           for yc in y_band_centers}

CHECKPOINT_DIR = OUTPUT_DIR + 'checkpoints_nb05/'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)


def checkpoint_path(y_c):
    return '{0}y_band_{1:.1f}.pkl'.format(CHECKPOINT_DIR, y_c)


mlt_all, clim_all, thresh_all, budget_all, bclim_all = {}, {}, {}, {}, {}

for y_c, band in y_bands.items():
    if not band:
        continue
    ckpt = checkpoint_path(y_c)
    if os.path.exists(ckpt):
        print('-- y band {0:.0f} ({1} tiles) - from checkpoint'.format(y_c, len(band)))
        with open(ckpt, 'rb') as fh:
            saved = pickle.load(fh)
        for d, key in ((mlt_all, 'mlt'), (clim_all, 'clim'), (thresh_all, 'thresh'),
                       (budget_all, 'budget'), (bclim_all, 'bclim')):
            d.update(saved[key])
        continue

    print('-- y band {0:.0f} ({1} tiles) --'.format(y_c, len(band)), flush=True)

    mlt_lazy    = {bid: _box_mean(temp_wide,        b) for bid, b in band}
    clim_lazy   = {bid: _box_mean(clim_wide.temp,   b) for bid, b in band}
    thresh_lazy = {bid: _box_mean(thresh_wide.temp, b) for bid, b in band}
    print('   MLT / clim / threshold ...', flush=True)
    mlt_band    = dict(xr.Dataset(mlt_lazy).compute())
    clim_band   = dict(xr.Dataset(clim_lazy).compute())
    thresh_band = dict(xr.Dataset(thresh_lazy).compute())

    budget_lazy, bclim_lazy = {}, {}
    for bid, b in band:
        sub = _box_slice(budget_wide, b)
        bc  = _box_slice(budget_clim_wide, b)
        for term, raw in TERM_MAP.items():
            budget_lazy['{0}__{1}'.format(bid, term)] = (
                sub[raw].weighted(_weights(sub)).mean(('yt_ocean', 'xt_ocean')) * SEC_PER_DAY)
            bclim_lazy['{0}__{1}'.format(bid, term)] = (
                bc[raw].weighted(_weights(bc)).mean(('yt_ocean', 'xt_ocean')) * SEC_PER_DAY)
    print('   budget series ...', flush=True)
    budget_band = dict(xr.Dataset(budget_lazy).compute())
    print('   budget climatology ...', flush=True)
    bclim_band  = dict(xr.Dataset(bclim_lazy).compute())

    with open(ckpt, 'wb') as fh:
        pickle.dump({'mlt': mlt_band, 'clim': clim_band, 'thresh': thresh_band,
                     'budget': budget_band, 'bclim': bclim_band}, fh)

    mlt_all.update(mlt_band);       clim_all.update(clim_band)
    thresh_all.update(thresh_band); budget_all.update(budget_band)
    bclim_all.update(bclim_band)

print('\nPre-compute done: {0} tiles'.format(len(mlt_all)))

## 5. Detect events and decompose the budget

Identical to NB04 section 9. The budget climatology is already a smoothed daily seasonal cycle
on a `dayofyear` axis, so it is placed on the time axis by selection — no interpolation.

In [ ]:
def detect_mhw_events(temp_da, thresh_da, min_dur=5, max_gap=2):
    """Hobday et al. (2016) event detection. Returns dicts: t_start, t_peak, t_end."""
    t      = pd.to_datetime(temp_da.time.values)
    y_temp = temp_da.values
    y_thr  = thresh_da.values
    valid  = np.isfinite(y_temp) & np.isfinite(y_thr)
    if valid.sum() < min_dur:
        return []

    exceed = pd.Series((y_temp[valid] > y_thr[valid]).astype(int), index=t[valid])
    runs        = (exceed.diff(1).ne(0)).cumsum()
    run_lengths = exceed.groupby(runs).transform('size')
    merged      = exceed.copy()
    merged[(exceed == 0) & (run_lengths <= max_gap)] = 1

    runs2     = (merged.diff(1).ne(0)).cumsum()
    lens2     = merged.groupby(runs2).transform('size')
    long_true = (merged == 1) & (lens2 >= min_dur)

    anom = pd.Series(y_temp[valid] - y_thr[valid], index=t[valid])
    arr, idx = long_true.to_numpy(), long_true.index.to_numpy()
    events, start_i = [], None
    for i in range(len(arr)):
        if arr[i] and start_i is None:
            start_i = i
        if start_i is not None and (i == len(arr) - 1 or not arr[i + 1]):
            seg = anom.iloc[start_i:i + 1]
            events.append({'t_start': idx[start_i], 't_peak': seg.idxmax(), 't_end': idx[i]})
            start_i = None
    return events


def get_season(timestamp):
    """Austral season from the month of the timestamp."""
    m = pd.Timestamp(timestamp).month
    if m in (12, 1, 2):
        return 'DJF'
    if m in (3, 4, 5):
        return 'MAM'
    if m in (6, 7, 8):
        return 'JJA'
    return 'SON'


all_events, all_df = {}, {}

for k, (bid, box) in enumerate(zip(box_ids, all_boxes)):
    if k % 20 == 0:
        print('[{0:3d}/{1}] {2}'.format(k, len(all_boxes), bid), flush=True)

    temp_vals   = mlt_all[bid]
    clim_mean   = clim_all[bid]
    thresh_mean = thresh_all[bid]

    doy_all        = temp_vals.time.dt.dayofyear.clip(max=int(clim_mean.dayofyear.max()))
    thresh_on_time = thresh_mean.sel(dayofyear=doy_all).assign_coords(time=temp_vals.time)

    events = detect_mhw_events(temp_vals, thresh_on_time, MIN_DUR, MAX_GAP)
    all_events[bid] = events
    if not events:
        all_df[bid] = pd.DataFrame()
        continue

    budget_series = xr.Dataset(
        {term: budget_all['{0}__{1}'.format(bid, term)] for term in BUDGET_TERMS})
    bc_box = xr.Dataset(
        {term: bclim_all['{0}__{1}'.format(bid, term)] for term in BUDGET_TERMS})

    doy_b = budget_series.time.dt.dayofyear.clip(max=int(bc_box.dayofyear.max()))
    clim_daily = xr.Dataset({
        name: (bc_box[name].sel(dayofyear=doy_b)
               .drop_vars('dayofyear')
               .assign_coords(time=budget_series.time))
        for name in BUDGET_TERMS})
    anom_series = xr.Dataset(
        {name: budget_series[name] - clim_daily[name] for name in BUDGET_TERMS})

    rows = []
    for ev in events:
        t_s, t_p, t_e = ev['t_start'], ev['t_peak'], ev['t_end']
        season = get_season(t_p)
        for name in BUDGET_TERMS:
            rows.append({
                't_start': t_s, 't_peak': t_p, 't_end': t_e,
                'season': season, 'term': name,
                'onset':   float(anom_series[name].sel(time=slice(t_s, t_p)).mean()),
                'decline': float(anom_series[name].sel(time=slice(t_p, t_e)).mean()),
            })
    all_df[bid] = pd.DataFrame(rows)

total_events = sum(len(v) for v in all_events.values())
print('\nDone. {0} events across {1} tiles.'.format(total_events, len(all_boxes)))

## 6. Reshape to `ds_events`

NB04's `box_df2xr`, unchanged: the long per-event table becomes a dataset with dimensions
(events × tiles × phase), one variable per budget term, tile centre coordinates attached.

In [ ]:
def box_df2xr(df):
    """Long per-event table -> wide dataset with dims (events, phase)."""
    event_keys = ['t_start', 't_peak', 't_end']
    df = df.copy()
    df['event_id'] = df.groupby(event_keys).ngroup()

    long = (df.melt(id_vars=['event_id'] + event_keys + ['term'],
                    value_vars=['onset', 'decline'],
                    var_name='phase', value_name='value')
              .pivot_table(index=['event_id', 'phase'], columns='term', values='value'))

    ds = long.to_xarray().rename({'event_id': 'events'})
    meta = df[['event_id'] + event_keys].drop_duplicates().set_index('event_id')
    ds = ds.assign_coords(t_start=('events', meta['t_start'].values),
                          t_peak=('events', meta['t_peak'].values),
                          t_end=('events', meta['t_end'].values))
    ds = ds.transpose('events', 'phase')
    for var in list(ds.data_vars):
        ds[var] = ds[var].assign_attrs({'standard_name': var,
                                        'units': 'degC day$^{-1}$'})
        ds = ds.rename({var: TERM_MAP[var]})
    return ds


tmp_ds = [box_df2xr(df).expand_dims({'tiles': [bid]})
          for bid, df in all_df.items() if not df.empty]
ds_events = xr.concat(tmp_ds, 'tiles', join='outer', coords='different')

# Attach tile centres, in index space and in degrees
box_by_id = dict(zip(box_ids, all_boxes))
ds_events = ds_events.assign_coords(
    x_center=('tiles', [box_by_id[t]['x_center'] for t in ds_events.tiles.values]),
    y_center=('tiles', [box_by_id[t]['y_center'] for t in ds_events.tiles.values]),
    lat_center=('tiles', [box_by_id[t]['lat_center'] for t in ds_events.tiles.values]),
    lon_center=('tiles', [box_by_id[t]['lon_center'] for t in ds_events.tiles.values]),
)
ds_events.attrs.update({
    'description': 'Antarctic MHW mixed-layer heat budget, Richaud-method tiles',
    'model': 'ACCESS-OM2 0.25 deg IAF cycle 6',
    'outputs': '{0}-{1}'.format(START_OUTPUT, END_OUTPUT),
    'box_size_cells': BOX_SIZE,
    'min_ocn_frac': MIN_OCN_FRAC,
    'mhw_min_dur': MIN_DUR,
    'mhw_max_gap': MAX_GAP,
    'budget_clim': BUDGET_CLIM_FILE,
})

out_path = OUTPUT_DIR + '05_Antarctic_mhw_budget_events.nc'
ds_events.to_netcdf(out_path)
print('Saved -> {0}'.format(out_path))
print(ds_events)

## 7. Driver ranking

NB04 cells 39–40. For every (tile, event) the four driver terms are ranked; `df_*_rank` then
counts, for each rank, how many events had each term at that rank.

`RANK_BY = 'signed'` reproduces NB04. `'abs'` ranks by magnitude instead, which is what NB03
does — see the header note on why they differ during decline.

In [ ]:
DRIVER_TERMS = [t for t in TERM_MAP.values() if t != 'mlt_tendency']


def phase_ranks(ds_events, phase, rank_by=RANK_BY):
    """(tiles x events) table of driver ranks for one phase. 1 = primary."""
    df = (ds_events.sel(phase=phase).drop_vars(['phase'])
          .to_dataframe(dim_order=('tiles', 'events'))
          .dropna()[DRIVER_TERMS])
    if rank_by == 'abs':
        df = df.abs()
    return df.rank(axis=1, ascending=False)


ds_rank_onset   = phase_ranks(ds_events, 'onset')
ds_rank_decline = phase_ranks(ds_events, 'decline')

n_rank = len(DRIVER_TERMS)
df_Onset_rank   = pd.DataFrame(columns=DRIVER_TERMS, index=np.arange(1, n_rank + 1))
df_Decline_rank = pd.DataFrame(columns=DRIVER_TERMS, index=np.arange(1, n_rank + 1))
for rk in np.arange(1, n_rank + 1):
    df_Onset_rank.loc[rk]   = ds_rank_onset.where(ds_rank_onset == rk).count().values
    df_Decline_rank.loc[rk] = ds_rank_decline.where(ds_rank_decline == rk).count().values
df_Onset_rank   = df_Onset_rank.astype('float')
df_Decline_rank = df_Decline_rank.astype('float')

nMHW = df_Onset_rank.sum()['advection']
print('Events ranked: {0:.0f}   (ranking by {1})'.format(nMHW, RANK_BY))
print('\nOnset — % of events at each rank:')
print((df_Onset_rank / nMHW * 100).round(1))
print('\nDecline — % of events at each rank:')
print((df_Decline_rank / nMHW * 100).round(1))

## 8. Driver bar chart

NB04 cell 42 (itself a reproduction of Fig. 4 of the 2024 paper), rebuilt in matplotlib —
the four colours are NB04's, by their XKCD names, so the two hemispheres match. Rows are onset and decline,
columns are the primary and secondary driver, bars are the percentage of events for which each
term held that rank.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 7.5), sharey=True)
panels = [(0, 0, df_Onset_rank, 1), (0, 1, df_Onset_rank, 2),
          (1, 0, df_Decline_rank, 1), (1, 1, df_Decline_rank, 2)]

for n, (r, c, df, rank) in enumerate(panels):
    ax = axes[r, c]
    ax.bar(np.arange(len(DRIVER_TERMS)), df.loc[rank].values / nMHW * 100,
           color=DRIVER_COLORS, edgecolor='k', linewidth=0.4)
    ax.set_ylim(0, 100)
    ax.yaxis.set_major_formatter(PercentFormatter())
    ax.set_xticks(np.arange(len(DRIVER_TERMS)))
    ax.set_xticklabels(DRIVER_TICKS if r == 1 else [], rotation=30, ha='right')
    ax.text(0.02, 0.97, 'abcd'[n] + ')', transform=ax.transAxes,
            va='top', ha='left', fontsize=12, fontweight='bold')
    ax.grid(axis='y', alpha=0.3, linewidth=0.5)
    if c == 1:
        sec = ax.secondary_yaxis(
            'right', functions=(lambda x: x / 100 * nMHW, lambda y: y / nMHW * 100))
        sec.set_ylabel('Number of MHWs driven [-]')

for c, lab in enumerate(['Primary', 'Secondary']):
    axes[0, c].set_title(lab, fontsize=13)
for r, lab in enumerate(['Onset', 'Decline']):
    axes[r, 0].annotate(lab, xy=(-0.16, 0.5), xycoords='axes fraction', rotation=90,
                        va='center', ha='center', fontsize=13, fontweight='bold')
fig.supylabel('Percentage of Marine Heatwaves driven [%]', fontsize=11)

handles = [plt.Rectangle((0, 0), 1, 1, facecolor=cc, edgecolor='k', linewidth=0.4)
           for cc in DRIVER_COLORS]
fig.legend(handles, DRIVER_LEGEND, loc='lower center', ncol=4, frameon=False,
           bbox_to_anchor=(0.5, 0.005))
fig.suptitle('Drivers for Antarctic Marine Heatwaves', fontsize=14)
fig.tight_layout(rect=[0.06, 0.07, 1, 0.96])
# fig.savefig(p2figs + 'BarPlot_allMHWDrivers_ranking_Antarctic_v1.png', dpi=600)


## 9. Where each driver dominates

NB04 cells 43–44, rebuilt in matplotlib/cartopy on a south-polar projection. Marker size is the fraction of that tile's events
whose **primary** driver is the column's term. White markers mark every tile, so tiles with no
events of that kind are distinguishable from tiles with no data.

In [ ]:
CoordsT = ds_events[['lat_center', 'lon_center']]
MHWs_tile = ds_rank_onset.dropna().groupby('tiles').size()


def primary_fraction(rank_df, term):
    """Fraction of each tile's events for which `term` is the primary driver."""
    n = rank_df.where(rank_df[term] == 1).dropna().groupby('tiles').size()
    frac = (n / MHWs_tile).dropna()
    return frac, CoordsT.sel(tiles=frac.index)


frac_onset   = {t: primary_fraction(ds_rank_onset, t)   for t in DRIVER_TERMS}
frac_decline = {t: primary_fraction(ds_rank_decline, t) for t in DRIVER_TERMS}
print('Tiles with events: {0}'.format(len(MHWs_tile)))
for t in DRIVER_TERMS:
    print('  {0:14s} primary in {1:3d} tiles (onset), {2:3d} (decline)'.format(
        t, len(frac_onset[t][0]), len(frac_decline[t][0])))

In [ ]:
ORDER = ['surf_to_ML', 'advection', 'vert_mixing', 'entrainment']

fig, axes = plt.subplots(2, 4, figsize=(18, 9),
                         subplot_kw={'projection': ccrs.SouthPolarStereo()})
sc = None
for j, term in enumerate(ORDER):
    for row, fracs in ((0, frac_onset), (1, frac_decline)):
        ax = axes[row, j]
        polar_ax(ax)
        # every tile in white first, so "no events of this kind" is visibly
        # different from "no data"
        ax.scatter(CoordsT.lon_center, CoordsT.lat_center, s=42, c='w',
                   edgecolor='w', transform=ccrs.PlateCarree(), zorder=5)
        frac, coords = fracs[term]
        h = ax.scatter(coords.lon_center, coords.lat_center,
                       s=frac.values * SMAX, c=DRIVER_COLORS[j], edgecolor='none',
                       transform=ccrs.PlateCarree(), zorder=6)
        if sc is None:
            sc = h
        if row == 0:
            ax.set_title(DRIVER_TITLES[j].replace('\n', ' '), fontsize=12)
        ax.text(0.02, 0.97, 'abcdefgh'[row * 4 + j] + ')', transform=ax.transAxes,
                va='top', fontsize=11, fontweight='bold')

for row, lab in enumerate(['Onset', 'Decline']):
    axes[row, 0].annotate(lab, xy=(-0.10, 0.5), xycoords='axes fraction', rotation=90,
                          va='center', ha='center', fontsize=13, fontweight='bold')

hs, ls = sc.legend_elements(prop='sizes', num=4, color='0.4',
                            func=lambda s: s / SMAX * 100, fmt='{x:.0f}%')
fig.legend(hs, ls, loc='lower center', ncol=len(hs), frameon=False,
           title='Fraction of MHWs driven', bbox_to_anchor=(0.5, 0.0))
fig.suptitle('Location of Antarctic Marine Heatwaves dominated by:', fontsize=14)
fig.tight_layout(rect=[0.02, 0.07, 1, 0.95])
# fig.savefig(p2figs + 'Map_MHWDrivers_primary_Antarctic_v1.png', dpi=600)


## 10. Comparing with NB03

Both notebooks now decompose the same events over the same period from the same budget
climatology, so a disagreement between them is about *method*, not data:

- **Tiles vs boxes.** 20×20 grid cells here (≈5° of longitude at 60°S, narrowing poleward)
  against 2.5°×2.5° geographic boxes in NB03. Tile area is roughly constant here; box area
  shrinks poleward there.
- **Counts vs composites.** This notebook counts events by which term ranks first; NB03
  averages the anomaly per term. A term can dominate most events while another carries the
  larger mean anomaly — that is a real result, not a contradiction.
- **Signed vs absolute ranking**, as set out in the header.

`05_Antarctic_mhw_budget_events.nc` holds the per-event values, so anything NB03 reports can be
recomputed from it without rerunning the detection.

In [ ]:
client.close()